# Advanced Wi-Fi Track Data Clean

This method re-clean data according to track repeat frequency ( caused by unstable signal appears among multiple trackers) based on given basic cleaned data.

Generate **virtual wifi tracker** after cleaning.(Virtual wifi trackers' id will be under zero)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mat
import os
import datetime
import time
from tqdm import tqdm
import plotly.express as px

mat.rcParams['font.family'] = 'SimHei'
mat.rcParams['font.sans-serif'] = 'SimHei'

import warnings
warnings.filterwarnings('ignore')

### Get wifi track positions

In [5]:
df_wifipos = pd.read_csv(os.getcwd()+'\\wifi_track_pos&traj\\dacang\\wifi_pos.csv')
df_wifipos.set_index(['wifi'],inplace=True)
df_wifipos

,X,Y
wifi,,
66,13.949,16.375
96,62.374,59.313
105,142.814,97.329
33,194.916,125.425
99,239.844,158.207
5,287.395,193.229
95,323.590,208.748
88,376.773,212.234
19,102.488,48.308


In [14]:
fig = px.scatter(x=df_wifipos.X,y=df_wifipos.Y,
                 labels=dict(x='x相对坐标',y='y相对坐标'))
fig.update_layout(
    height=600,
    width = 800
)
fig.show()

### Read Wifi Track Data

In [19]:
df_read = pd.read_csv(os.getcwd()+'\\cleaned_data\\dacang\\dacang_tourist_data.csv')
df = df_read.copy()
df_read

,a,r,t,m,calender
0,22,-38,2022-07-05 23:52:23,"28,54,187,20,163,246",7.5
1,22,-37,2022-07-05 23:52:49,"28,54,187,20,163,246",7.5
2,22,-41,2022-07-05 23:53:02,"28,54,187,20,163,246",7.5
3,22,-42,2022-07-05 23:53:15,"28,54,187,20,163,246",7.5
4,22,-46,2022-07-05 23:53:28,"28,54,187,20,163,246",7.5
...,...,...,...,...,...
241406,32,-48,2022-08-07 23:49:21,"124,161,119,201,50,40",8.7
241407,32,-50,2022-08-07 23:50:13,"124,161,119,201,50,40",8.7
241408,32,-50,2022-08-07 23:50:52,"124,161,119,201,50,40",8.7
241409,32,-42,2022-08-07 23:51:44,"124,161,119,201,50,40",8.7


In [23]:
#label each mac with date
df['oriMac'] = df['m']
for index,row in df.iterrows():
    df.loc[index,'m'] = str(row['calender'])+'-'+row.m
df

,a,r,t,m,calender,oriMac
0,22,-38,2022-07-05 23:52:23,"7.5-28,54,187,20,163,246",7.5,"28,54,187,20,163,246"
1,22,-37,2022-07-05 23:52:49,"7.5-28,54,187,20,163,246",7.5,"28,54,187,20,163,246"
2,22,-41,2022-07-05 23:53:02,"7.5-28,54,187,20,163,246",7.5,"28,54,187,20,163,246"
3,22,-42,2022-07-05 23:53:15,"7.5-28,54,187,20,163,246",7.5,"28,54,187,20,163,246"
4,22,-46,2022-07-05 23:53:28,"7.5-28,54,187,20,163,246",7.5,"28,54,187,20,163,246"
...,...,...,...,...,...,...
241406,32,-48,2022-08-07 23:49:21,"8.7-124,161,119,201,50,40",8.7,"124,161,119,201,50,40"
241407,32,-50,2022-08-07 23:50:13,"8.7-124,161,119,201,50,40",8.7,"124,161,119,201,50,40"
241408,32,-50,2022-08-07 23:50:52,"8.7-124,161,119,201,50,40",8.7,"124,161,119,201,50,40"
241409,32,-42,2022-08-07 23:51:44,"8.7-124,161,119,201,50,40",8.7,"124,161,119,201,50,40"


In [24]:
len(df.m.unique())

16663

In [27]:
#删除打上标签后只出现过一次的mac
df = df[df.duplicated('m',keep=False)]
df

,a,r,t,m,calender,oriMac
0,22,-38,2022-07-05 23:52:23,"7.5-28,54,187,20,163,246",7.5,"28,54,187,20,163,246"
1,22,-37,2022-07-05 23:52:49,"7.5-28,54,187,20,163,246",7.5,"28,54,187,20,163,246"
2,22,-41,2022-07-05 23:53:02,"7.5-28,54,187,20,163,246",7.5,"28,54,187,20,163,246"
3,22,-42,2022-07-05 23:53:15,"7.5-28,54,187,20,163,246",7.5,"28,54,187,20,163,246"
4,22,-46,2022-07-05 23:53:28,"7.5-28,54,187,20,163,246",7.5,"28,54,187,20,163,246"
...,...,...,...,...,...,...
241406,32,-48,2022-08-07 23:49:21,"8.7-124,161,119,201,50,40",8.7,"124,161,119,201,50,40"
241407,32,-50,2022-08-07 23:50:13,"8.7-124,161,119,201,50,40",8.7,"124,161,119,201,50,40"
241408,32,-50,2022-08-07 23:50:52,"8.7-124,161,119,201,50,40",8.7,"124,161,119,201,50,40"
241409,32,-42,2022-08-07 23:51:44,"8.7-124,161,119,201,50,40",8.7,"124,161,119,201,50,40"


In [29]:
#获得只存在过一个地方的mac名单
list_count = df.groupby(['m']).a.value_counts()
dd = list_count.to_frame().rename(columns={'a': 'A'}).reset_index()
mac_Once = dd[~dd.duplicated('m', keep=False)].m.reset_index().drop('index',
                                                                    axis=1)
mac_Once

,m
0,"7.1-104,128,62,174,96,98"
1,"7.1-104,160,62,174,112,98"
2,"7.1-104,160,62,174,96,96"
3,"7.1-104,160,63,174,96,98"
4,"7.1-104,161,62,174,96,98"
...,...
2590,"8.7-80,210,245,240,209,10"
2591,"8.7-84,167,3,191,88,22"
2592,"8.7-96,145,243,118,168,36"
2593,"8.7-96,61,172,1,128,194"


In [37]:
mac_Once.iloc(0)

In [45]:
df[df.m == mac_Once.iloc[80].m]

,a,r,t,m,calender,oriMac
21827,84,-36,2022-07-10 19:16:38,"7.1-204,185,229,56,107,179",7.1,"204,185,229,56,107,179"
22046,84,-36,2022-07-10 12:54:04,"7.1-204,185,229,56,107,179",7.1,"204,185,229,56,107,179"


In [46]:
df[df.oriMac == '204,185,229,56,107,179']

,a,r,t,m,calender,oriMac
21827,84,-36,2022-07-10 19:16:38,"7.1-204,185,229,56,107,179",7.1,"204,185,229,56,107,179"
22046,84,-36,2022-07-10 12:54:04,"7.1-204,185,229,56,107,179",7.1,"204,185,229,56,107,179"
